# Lab 3.12 &mdash; Capstone: A Mini Transformer Pipeline

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Chain tokenize -> embed -> positional-encode -> self-attend -> pool
- Produce a single vector that represents a whole sentence
- Confirm different sentences get different representations

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 3 slides &mdash; The transformer block](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-03-12"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept &mdash; the module's payoff
Everything you built now snaps together into a **mini transformer encoder**: tokenize a sentence,
look up **embeddings**, add **positional encodings**, run **self-attention**, and **mean-pool** the
result into one sentence vector &mdash; the representation a classifier or search index would use.

In [ ]:
# DEMO -- the shared pieces (given)
import numpy as np
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)
def positional_encoding(seq_len, d):
    pos = np.arange(seq_len)[:, None]; i = np.arange(d)[None, :]
    ang = pos / np.power(10000, (2 * (i // 2)) / d)
    pe = np.zeros((seq_len, d)); pe[:, 0::2] = np.sin(ang[:, 0::2]); pe[:, 1::2] = np.cos(ang[:, 1::2])
    return pe
rng = np.random.default_rng(0)
VOCAB = {w: i for i, w in enumerate("the cat sat on mat dog ran fast".split())}
TABLE = rng.normal(size=(len(VOCAB), 4))   # an embedding per vocab word, d=4
print("vocab:", VOCAB)

## Your Turn
Complete the four pipeline steps.

In [ ]:
import numpy as np
def embed(tokens):
    return np.array([___ for t in tokens])   # TODO: look up TABLE[VOCAB[t]]

def encode_sentence(sentence):
    tokens = sentence.split()
    X = embed(tokens)
    d = X.shape[-1]
    X = X + ___   # TODO: add positional_encoding(len(tokens), d)
    A = ___   # TODO: softmax(X @ X^T / sqrt(d), axis=-1)
    context = A @ X
    return ___   # TODO: mean-pool the context over the tokens (axis=0)

try:
    v = encode_sentence('the cat ran')
    print('sentence vector (dim', len(v), '):', np.round(v, 3))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

import numpy as np
expect_true("embed returns one row per token", lambda: embed(["the", "cat"]).shape == (2, 4))
expect_true("encode_sentence returns a single d-dim vector", lambda: encode_sentence("the cat ran").shape == (4,))
expect_true("the pipeline is deterministic", lambda: np.allclose(encode_sentence("the cat ran"), encode_sentence("the cat ran")))
expect_true("different sentences -> different vectors", lambda: not np.allclose(encode_sentence("the cat ran"), encode_sentence("the dog sat")))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real thing with Hugging Face (not graded)
Compare your hand-built sentence vector to a real transformer's sentence embedding. Safe to skip &mdash; it needs `pip install transformers torch` and a one-time model
download. If `transformers` is not installed, the cell simply prints a note and moves on.

In [ ]:
try:
    from transformers import pipeline
    import numpy as np
    fx = pipeline("feature-extraction", model="prajjwal1/bert-tiny")
    emb = np.array(fx("the cat ran")[0]).mean(axis=0)
    print("real transformer sentence vector dim:", emb.shape)
except Exception as e:
    print("Optional real-model demo skipped (the graded cells above already covered this).")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
You built a working transformer encoder from parts you wrote yourself. Day 2 Module 4 takes the real, pretrained versions and fine-tunes them. That completes Module 3.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>